<a href="https://colab.research.google.com/github/ArjunHirani/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

In [1]:
# Setup: reconnect and rebuild the same March (feature) -> April (outcome) frame
# used in w03/w04, self-contained since this is a separate notebook file.
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

data = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS impressions_march,
               SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks_march,
               AVG(CASE WHEN gsc_data_available THEN gsc_avg_position END) AS avg_position_march,
               COUNT(DISTINCT CASE WHEN gsc_data_available THEN report_date END) AS active_days_march
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2
        HAVING impressions_march >= 100
    ),
    april AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks_april
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04'
        GROUP BY 1, 2
    )
    SELECT m.*, a.clicks_april
    FROM march m JOIN april a USING (client_hash_id, content_hash_id)
""").df()

content_meta = con.sql(f"SELECT client_hash_id, content_hash_id, content_created_date FROM {TABLES['dim_content']}").df()
data = data.merge(content_meta, on=['client_hash_id', 'content_hash_id'], how='left')
data['content_age_days_march'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(data['content_created_date'])).dt.days
data['ctr_march'] = data['clicks_march'] / data['impressions_march']
data['is_declining_april'] = (data['clicks_april'] < data['clicks_march']).astype(int)
data['position_tier'] = pd.cut(data['avg_position_march'], bins=[0, 3, 10, 20, 50, np.inf],
                                labels=['top3', '4-10', '11-20', '21-50', '50+'])
expected_ctr = data.groupby('position_tier', observed=True)['ctr_march'].transform('mean')
data['ctr_gap'] = data['ctr_march'] - expected_ctr

data = data.dropna(subset=['content_age_days_march', 'avg_position_march', 'ctr_march', 'ctr_gap']).reset_index(drop=True)
print(f"Rows: {len(data):,}  |  distinct clients: {data['client_hash_id'].nunique()}  |  base rate: {data['is_declining_april'].mean():.3f}")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 101,441  |  distinct clients: 44  |  base rate: 0.402


## 1. Method choice and why


**Logistic Regression, then Random Forest** — the method menu's own recommendation for a
yes/no question with an observed label (`is_declining_april`, built from real April clicks, not
a defined rule). I train both, not just one: Logistic Regression as the simple, fully readable
first pass, Random Forest as the stronger model that can pick up combinations of my weak
individual signals.

This choice is grounded directly in what Weeks 2-3 already found: no single signal (position,
CTR, staleness) correlated strongly with the outcome on its own, and a hand-combined rule with
equal weights (Week 4's baseline) can only combine signals the way a human guesses, not the way
the data actually weights them. A Random Forest earns its complexity here specifically because
the earlier weeks already showed the pattern is real but not linearly obvious — exactly the
situation the framing skill says ML should be reached for, not before.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-grouped split** (`GroupShuffleSplit` on `client_hash_id`), 75/25. This directly fixes a
limitation I flagged myself in Weeks 3-4: a random row-level split lets the same client's other
content items appear in both train and test, which lets a model partly learn "this client's
general level" rather than genuine per-page signal — a softer, client-level leakage risk. A
client-grouped split means every content item from a given client is entirely in train OR
entirely in test, never both, so the test score reflects generalization to **unseen clients**,
not just unseen pages from clients the model has already partly learned.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['impressions_march', 'clicks_march', 'avg_position_march',
                 'active_days_march', 'content_age_days_march', 'ctr_march', 'ctr_gap']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data['client_hash_id']))
train_df = data.iloc[train_idx].reset_index(drop=True)
test_df = data.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print(f"Train rows: {len(train_df):,}  ({train_df['client_hash_id'].nunique()} clients)")
print(f"Test rows:  {len(test_df):,}  ({test_df['client_hash_id'].nunique()} clients)")
print(f"Client overlap between train and test (should be EMPTY set): {overlap}")
assert len(overlap) == 0, "Client leakage across the split!"
print(f"\nTrain base rate: {train_df['is_declining_april'].mean():.3f}  |  Test base rate: {test_df['is_declining_april'].mean():.3f}")

Train rows: 93,085  (33 clients)
Test rows:  8,356  (11 clients)
Client overlap between train and test (should be EMPTY set): set()

Train base rate: 0.404  |  Test base rate: 0.376


## 3. Train + compare vs my baseline

The Week-4 baseline rule is recomputed fresh **within this same test set** (percentile ranks are
population-relative, so they must be computed on the exact population being scored), so every
row in the comparison table below is evaluated on identical data: the same 25% held-out clients,
the same Precision@20/50 metric, the same base rate.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_train, y_train = train_df[feature_cols], train_df['is_declining_april']
X_test, y_test = test_df[feature_cols], test_df['is_declining_april']

# --- Baseline rule, recomputed fresh on the TEST split only ---
test_df = test_df.copy()
staleness_rank = test_df['content_age_days_march'].rank(pct=True)
ctr_deficit_rank = (-test_df['ctr_gap']).rank(pct=True)
baseline_score_test = 0.5 * staleness_rank + 0.5 * ctr_deficit_rank

# --- Logistic Regression ---
logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
logreg_score_test = logreg.predict_proba(X_test)[:, 1]

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(X_train, y_train)
rf_score_test = rf.predict_proba(X_test)[:, 1]

base_rate_test = y_test.mean()
results = []
for name, scores in [('Baseline rule (Week 4)', baseline_score_test),
                       ('Logistic Regression', logreg_score_test),
                       ('Random Forest', rf_score_test)]:
    row = {
        'method': name,
        'precision_at_20': round(precision_at_k(scores, y_test.values, 20), 3),
        'precision_at_50': round(precision_at_k(scores, y_test.values, 50), 3),
        'roc_auc': round(roc_auc_score(y_test, scores), 3),
    }
    results.append(row)

comparison = pd.DataFrame(results)
comparison['base_rate'] = round(base_rate_test, 3)
print(comparison.to_string(index=False))

                method  precision_at_20  precision_at_50  roc_auc  base_rate
Baseline rule (Week 4)             0.00             0.00    0.285      0.376
   Logistic Regression             0.60             0.66    0.715      0.376
         Random Forest             0.95             0.96    0.856      0.376


**Reading the real comparison table:** the Week-4 baseline rule scores far worse than random
on this client-grouped test set — Precision@20 and Precision@50 both **0.00**, and ROC-AUC
**0.285** (below the 0.5 random-guessing line, meaning the rule points the wrong direction
here). Random Forest clearly wins on both metrics (Precision@50: 0.96 vs 0.00; Precision@20:
0.95 vs 0.00), and Logistic Regression already beats the baseline comfortably too
(Precision@50: 0.66, ROC-AUC 0.715).

Two honest things worth saying about that gap, not just celebrating it:

1. **The baseline isn't just weak here, it's inverted.** With only 44 distinct clients total,
   this 75/25 client-grouped split leaves just 11 clients in test — a small, specific slice.
   It's plausible the staleness/CTR-gap direction genuinely reverses for this particular client
   group, or that 11 clients is too few for the rule's signal to show reliably. I would not
   conclude "staleness and CTR-gap are useless" from this alone — I'd want to re-run with a few
   different random splits before trusting the direction, not just the magnitude.
2. **Compared to Week 3's honest 0.682 ROC-AUC** (Logistic Regression, random row-level split,
   5 features), this week's Logistic Regression on a proper client-grouped split scores
   **0.715** — slightly *higher*, not lower. That's reassuring: the client-level leakage
   caveat I flagged earlier doesn't seem to have inflated the original number much, since
   fixing it didn't collapse performance.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Permutation importance on the Random Forest (checked by shuffling, not just read off the fitted
tree) tells us what the model actually leans on — then a sanity check: does the top feature make
sense, or is it suspiciously perfect? And three concrete wrong cases, read by hand.

In [4]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=42, n_jobs=-1)
importance_table = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
print("Permutation importance (Random Forest, test set):")
print(importance_table.to_string(index=False))

# Three concrete wrong cases: highest-confidence FALSE POSITIVES and FALSE NEGATIVES
test_df['rf_score'] = rf_score_test
test_df['rf_pred'] = (test_df['rf_score'] >= 0.5).astype(int)
test_df['actual'] = y_test.values

false_positives = test_df[(test_df['rf_pred'] == 1) & (test_df['actual'] == 0)].sort_values('rf_score', ascending=False).head(2)
false_negatives = test_df[(test_df['rf_pred'] == 0) & (test_df['actual'] == 1)].sort_values('rf_score').head(1)

review_cols = ['content_hash_id', 'rf_score', 'impressions_march', 'clicks_march',
               'avg_position_march', 'ctr_gap', 'content_age_days_march', 'actual']
print("\nTop confident FALSE POSITIVES (model said decline, it didn't):")
print(false_positives[review_cols].to_string(index=False))
print("\nMost confident FALSE NEGATIVE (model said stable, it declined):")
print(false_negatives[review_cols].to_string(index=False))

Permutation importance (Random Forest, test set):
               feature  importance_mean  importance_std
             ctr_march         0.061632        0.003640
          clicks_march         0.059371        0.003499
     active_days_march         0.013840        0.001902
               ctr_gap         0.002005        0.000849
     impressions_march         0.001801        0.000958
    avg_position_march         0.000299        0.000634
content_age_days_march        -0.003183        0.001513

Top confident FALSE POSITIVES (model said decline, it didn't):
         content_hash_id  rf_score  impressions_march  clicks_march  avg_position_march  ctr_gap  content_age_days_march  actual
content_8a2dc3465ac8f497  0.783218              152.0           1.0           68.940746 0.006129                     180       0
content_1205d0d6921ea6df  0.753662              146.0           3.0           23.487628 0.019154                     236       0

Most confident FALSE NEGATIVE (model said stable, 

**What the model leans on:** the top two permutation-importance features are `ctr_march`
(0.062) and `clicks_march` (0.059) — both plausible, both fully March-observed. But there's a
subtlety worth naming, not just accepting: my label is `clicks_april < clicks_march`, a *strict*
inequality on raw counts. For pages with very few March clicks (1-2), a single April click is
enough to flip the label to "not declining" — a small-number artifact of how I defined the
label, not necessarily genuine SEO signal. Since `clicks_march` is the model's #2 feature, some
of Random Forest's very high precision (0.95-0.96) may partly reflect learning this low-volume
artifact rather than pure decline risk. This isn't true leakage (`clicks_march` is genuinely
March-only and passes Section 2's client-split audit) — it's a construct-validity concern about
the label definition itself, in the same family as Week 3's `scroll_rate` catch. A cleaner label
for later work would use a percentage-change threshold rather than a strict "<".

**The 3 wrong cases:**
- Both false positives (`content_8a2dc3465ac8f497`, `content_1205d0d6921ea6df`) are low-volume
  pages (152/146 March impressions, 1-3 March clicks) at a mediocre position — the model
  expected decline given volume/position, but they held steady. Plausibly the small-number
  effect above: with only 1-3 March clicks, whether April clicks meet that bar is close to a
  coin flip.
- The false negative (`content_9cde9aebfea83b32`) is the opposite story: a strong page by every
  feature here — 1,279 impressions, 31 clicks, good position (3.8), a positive CTR gap — yet it
  declined anyway. This is the most informative miss: nothing in these 7 features warned of it,
  meaning something outside this feature set (a competitor, a SERP change, seasonality) drives
  real variance no March-only snapshot can see.